In [ ]:
"""Train a baseline gradient-boosting model per horizon and write predictions.parquet.

Baseline approach: histogram GBM regression (sklearn's LightGBM-equivalent)
on each target (target_10d, target_30d), with a time-based holdout for a
sanity-check Spearman score. Predictions are rank-normalized to [0, 1]
per the submission spec (id, pred_10d, pred_30d).
"""

from pathlib import Path

import polars as pl
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet, LinearRegression

DATA_DIR = Path("data") # folder
OUT_DIR = Path("predictions") # folder
OUT_DIR.mkdir(exist_ok=True)

train = pl.read_parquet(DATA_DIR / "training_data.parquet")
infer = pl.read_parquet(DATA_DIR / "inference_data.parquet")

feature_cols = [c for c in train.columns if c.startswith("feature_")] # extract the feature columns
print(f"{len(feature_cols)} features, {train.height:,} training rows, {infer.height} inference rows")

In [7]:
def ts_split(train, lookback):
    # time-based holdout: last 60 dates for validation
    dates = train["date"].unique().sort()
    split_date = dates[-lookback] # validation holdout
    tr = train.filter(pl.col("date") < split_date) # filter up to last training_date
    va = train.filter(pl.col("date") >= split_date) # validation date and beyond
    return tr, va

In [5]:
def HGBR(tr, va, preds):
    for target in ["target_10d", "target_30d"]: # iterates through both target variables
        tr_t = tr.drop_nulls(subset=[target]) # clean training data
        va_t = va.drop_nulls(subset=[target]) # clean val/test data for Nan's

        model = HistGradientBoostingRegressor(
            max_iter=500,
            learning_rate=0.02,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=42,
        )
        model.fit(tr_t[feature_cols].to_numpy(), tr_t[target].to_numpy()) # fit the model on training, converted to numpy arrays

        val_pred = model.predict(va_t[feature_cols].to_numpy()) # prediction of 10d and 30d
        corr, _ = spearmanr(va_t[target].to_numpy(), val_pred) # compare prediction from val and actual val target
        print(f"{target}: holdout Spearman = {corr:.4f}")

        # retrain on all data before predicting the live universe
        full = train.drop_nulls(subset=[target])
        model.fit(full[feature_cols].to_numpy(), full[target].to_numpy())
        raw = model.predict(infer[feature_cols].to_numpy())

        horizon = target.replace("target", "pred")
        preds['HGBR'][horizon] = pl.Series(raw).rank() / len(raw)  # rank-normalize to (0, 1]

    return preds # returns the predictions


In [19]:
def Ridge_regression(tr, va, preds):
    for target in ["target_10d", "target_30d"]: # iterates through both target variables
        tr_t = tr.drop_nulls(subset=[target]) # clean training data
        va_t = va.drop_nulls(subset=[target]) # clean val/test data for Nan's

        model = Ridge(alpha=1.0, random_state=42)
        model.fit(tr_t[feature_cols].to_numpy(), tr_t[target].to_numpy()) # fit the model on training, converted to numpy arrays

        val_pred = model.predict(va_t[feature_cols].to_numpy()) # prediction of 10d and 30d
        corr, _ = spearmanr(va_t[target].to_numpy(), val_pred) # compare prediction from val and actual val target
        print(f"{target}: holdout Spearman = {corr:.4f}")

        # retrain on all data before predicting the live universe
        full = train.drop_nulls(subset=[target])
        model.fit(full[feature_cols].to_numpy(), full[target].to_numpy())
        raw = model.predict(infer[feature_cols].to_numpy())

        horizon = target.replace("target", "pred")
        preds['Ridge'][horizon] = pl.Series(raw).rank() / len(raw)  # rank-normalize to (0, 1]

    return preds # returns the predictions

In [ ]:
def ElasticNet_regression(tr, va, preds):
    for target in ["target_10d", "target_30d"]: # iterates through both target variables
        tr_t = tr.drop_nulls(subset=[target]) # clean training data
        va_t = va.drop_nulls(subset=[target]) # clean val/test data for Nan's

        # alpha is much smaller than Ridge's: ElasticNet's L1 part zeroes out
        # coefficients aggressively, and at alpha=1.0 it would kill all 180
        model = ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42)
        model.fit(tr_t[feature_cols].to_numpy(), tr_t[target].to_numpy()) # fit the model on training, converted to numpy arrays

        val_pred = model.predict(va_t[feature_cols].to_numpy()) # prediction of 10d and 30d
        corr, _ = spearmanr(va_t[target].to_numpy(), val_pred) # compare prediction from val and actual val target
        print(f"{target}: holdout Spearman = {corr:.4f}")

        # retrain on all data before predicting the live universe
        full = train.drop_nulls(subset=[target])
        model.fit(full[feature_cols].to_numpy(), full[target].to_numpy())
        raw = model.predict(infer[feature_cols].to_numpy())

        horizon = target.replace("target", "pred")
        preds['ElasticNet'][horizon] = pl.Series(raw).rank() / len(raw)  # rank-normalize to (0, 1]

    return preds # returns the predictions

In [ ]:
def Linear_regression(tr, va, preds):
    for target in ["target_10d", "target_30d"]: # iterates through both target variables
        tr_t = tr.drop_nulls(subset=[target]) # clean training data
        va_t = va.drop_nulls(subset=[target]) # clean val/test data for Nan's

        model = LinearRegression() # plain OLS - no regularization, no knobs
        model.fit(tr_t[feature_cols].to_numpy(), tr_t[target].to_numpy()) # fit the model on training, converted to numpy arrays

        val_pred = model.predict(va_t[feature_cols].to_numpy()) # prediction of 10d and 30d
        corr, _ = spearmanr(va_t[target].to_numpy(), val_pred) # compare prediction from val and actual val target
        print(f"{target}: holdout Spearman = {corr:.4f}")

        # retrain on all data before predicting the live universe
        full = train.drop_nulls(subset=[target])
        model.fit(full[feature_cols].to_numpy(), full[target].to_numpy())
        raw = model.predict(infer[feature_cols].to_numpy())

        horizon = target.replace("target", "pred")
        preds['LR'][horizon] = pl.Series(raw).rank() / len(raw)  # rank-normalize to (0, 1]

    return preds # returns the predictions

In [ ]:
lookback = 60
MODELS = ["Ridge", "HGBR", "ElasticNet", "LR"] # one key per model
preds = {name: {"id": infer["id"]} for name in MODELS} # predictions

tr, va = ts_split(train, lookback)
print("-- Ridge --")
preds = Ridge_regression(tr, va, preds)
print("-- ElasticNet --")
preds = ElasticNet_regression(tr, va, preds)
print("-- LinearRegression --")
preds = Linear_regression(tr, va, preds)
print("-- HGBR --")
final = HGBR(tr, va, preds)

In [ ]:
# Convert the predictions for display/inspection (better structure)
# one column per model+horizon, joined on 'id' - ready for aggregation later
df = None
for name in MODELS:
    df_m = pl.DataFrame({
        "id": final[name]["id"],
        f"{name}_pred_10d": final[name]["pred_10d"],
        f"{name}_pred_30d": final[name]["pred_30d"],
    })
    df = df_m if df is None else df.join(df_m, on="id", how="inner")

df.head()

In [34]:
df

id,Ridge_pred_10d,Ridge_pred_30d,HGBR_pred_10d,HGBR_pred_30d
str,f64,f64,f64,f64
"""0G""",0.294118,0.388235,0.135294,0.105882
"""2Z""",0.070588,0.123529,0.088235,0.141176
"""AAVE""",0.770588,0.829412,0.764706,0.641176
"""ACE""",0.029412,0.029412,0.005882,0.005882
"""ADA""",0.564706,0.623529,0.447059,0.729412
…,…,…,…,…
"""kFLOKI""",0.641176,0.652941,0.664706,0.582353
"""kLUNC""",0.476471,0.664706,0.676471,0.529412
"""kNEIRO""",0.023529,0.017647,0.188235,0.182353


In [ ]:
def aggregate(final):
    # equal-weight ensemble: average the four models' rank columns per horizon,
    # then re-rank the average back to (0, 1] so the output is a valid submission
    agg = {"id": final[MODELS[0]]["id"]} # ids are identical across models
    for horizon in ["pred_10d", "pred_30d"]:
        stacked = pl.DataFrame({name: final[name][horizon] for name in MODELS}) # one column per model
        mean_rank = stacked.mean_horizontal() # average the 4 rank predictions per asset
        agg[horizon] = mean_rank.rank() / len(mean_rank) # re-rank to (0, 1]
    return pl.DataFrame(agg) # submission format: id, pred_10d, pred_30d

In [ ]:
submission = aggregate(final) # final targets: aggregate pred_10d and pred_30d

# sanity checks against the submission spec before writing
assert submission.columns == ["id", "pred_10d", "pred_30d"] # exact required columns
assert submission["pred_10d"].is_between(0, 1).all() # floats in [0, 1]
assert submission["pred_30d"].is_between(0, 1).all()
assert submission.height >= 80 # minimum 80 assets

submission.write_parquet(OUT_DIR / "predictions.parquet") # wrap it up in a parquet
print(f"wrote {OUT_DIR / 'predictions.parquet'} ({submission.height} assets)")
submission.head()